In [1]:
# Import Necessary Libraries
import pandas as pd
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import joblib

In [2]:
# Load the dataset
BASE_DIR = os.getcwd()
employee_master = pd.read_csv(os.path.join(BASE_DIR,'..' ,'data','processed', 'employee_master_dataset.csv'))

In [3]:
# Import the preprocessor
preprocessor = joblib.load(os.path.join(BASE_DIR,'..' ,'models', 'preprocessor.pkl'))

In [4]:
# separate features and target variable
X = employee_master.drop('Attrition', axis=1)  # Separating features
Y = employee_master['Attrition'].map({'Yes': 1, 'No': 0})  # Encoding target variable
#Train test split
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.2, random_state=42,stratify=Y)

In [5]:
binary_cols = ['OverTime']
onehot_cols = ['Department', 'EducationField', 'JobRole', 'MaritalStatus']
ordinal_cols_business_travel = ['BusinessTravel']
ordinal_cols_scale = ['JobLevel', 'EnvironmentSatisfaction', 'JobSatisfaction',
                      'WorkLifeBalance', 'JobInvolvement', 'RelationshipSatisfaction',
                      'StockOptionLevel']
numerical_features = [col for col in X.columns 
                      if col not in binary_cols + onehot_cols + 
                      ordinal_cols_business_travel + ordinal_cols_scale]

print(f"Binary: {binary_cols}")
print(f"OneHot: {onehot_cols}")
print(f"Ordinal travel: {ordinal_cols_business_travel}")
print(f"Ordinal scale: {ordinal_cols_scale}")
print(f"Numerical: {numerical_features}")
print(f"Total assigned: {len(binary_cols + onehot_cols + ordinal_cols_business_travel + ordinal_cols_scale + numerical_features)}")

Binary: ['OverTime']
OneHot: ['Department', 'EducationField', 'JobRole', 'MaritalStatus']
Ordinal travel: ['BusinessTravel']
Ordinal scale: ['JobLevel', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'RelationshipSatisfaction', 'StockOptionLevel']
Numerical: ['Age', 'DistanceFromHome', 'Education', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'salary_growth_rate', 'bonus_month_ratio', 'overtime_month_ratio', 'avg_bonus_pct', 'career_stagnation_index', 'promotion_velocity', 'role_stagnation_ratio', 'manager_stability_ratio', 'training_intensity']
Total assigned: 31


In [6]:
# Creating a pipeline that includes Logistic Regression Classifier and Preprocessor
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',
        LogisticRegression(
            solver='lbfgs',
            class_weight='balanced',
            max_iter=5000,
            random_state=42
        )
    )
])

# Hyperparameter Grid
lr_param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['lbfgs', 'liblinear']
}


# Grid Search
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Train
lr_grid.fit(X_train, y_train)

# Evaluate
lr_train_auc = roc_auc_score(
    y_train,
    lr_grid.predict_proba(X_train)[:, 1]
)

lr_test_auc = roc_auc_score(
    y_test,
    lr_grid.predict_proba(X_test)[:, 1]
)

print(f"Logistic Regression Best Params: {lr_grid.best_params_}")
print(
    f"Logistic Regression → Train AUC: {lr_train_auc:.3f} | "
    f"Test AUC: {lr_test_auc:.3f} | "
    f"Gap: {lr_train_auc - lr_test_auc:.3f}"
)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


Logistic Regression Best Params: {'classifier__C': 0.1, 'classifier__solver': 'lbfgs'}
Logistic Regression → Train AUC: 0.880 | Test AUC: 0.801 | Gap: 0.079


In [7]:
# Creating a pipeline that includes XGBoost Classifier and Preprocessor
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    ))
])

# Hyperparameter Grid for XGBoost
xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [2, 3, 4],
    'classifier__learning_rate': [0.01, 0.05],
    'classifier__subsample': [0.6, 0.8],
    'classifier__colsample_bytree': [0.6, 0.8],
    'classifier__min_child_weight': [5, 10, 20],
    'classifier__reg_alpha': [0.1, 1.0],
    'classifier__reg_lambda': [1.0, 5.0],
    'classifier__scale_pos_weight': [5]
}

# Grid Search for XGBoost
xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# Train the XGBoost model
xgb_grid.fit(X_train, y_train)

# Evaluate the XGBoost model
xgb_best = xgb_grid.best_estimator_
xgb_train_auc = roc_auc_score(y_train, xgb_best.predict_proba(X_train)[:, 1])
xgb_test_auc = roc_auc_score(y_test, xgb_best.predict_proba(X_test)[:, 1])

print(f"Best params: {xgb_grid.best_params_}")
print(f"Best CV AUC: {xgb_grid.best_score_:.3f}")
print(f"Train AUC:   {xgb_train_auc:.3f}")
print(f"Test AUC:    {xgb_test_auc:.3f}")
print(f"Gap:         {xgb_train_auc - xgb_test_auc:.3f}")

Fitting 5 folds for each of 864 candidates, totalling 4320 fits
Best params: {'classifier__colsample_bytree': 0.6, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 2, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 300, 'classifier__reg_alpha': 0.1, 'classifier__reg_lambda': 1.0, 'classifier__scale_pos_weight': 5, 'classifier__subsample': 0.8}
Best CV AUC: 0.830
Train AUC:   0.966
Test AUC:    0.817
Gap:         0.148


In [8]:
# Save the best model
joblib.dump(xgb_best, os.path.join(BASE_DIR,'..' ,'models', 'best_model.pkl'))

['c:\\Projects\\TalentSight\\talentsight\\notebooks\\..\\models\\best_model.pkl']